# 06 · LLM response types: text, JSON you can trust, and streaming

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** get answers your *code* can use, not just text to read. You will control length, ask for JSON, survive messy JSON, and read a streamed answer.

**Done means:** a system prompt shapes the answer · you detect a cut-off answer · a messy reply is parsed and validated (with one retry) · a streamed answer is assembled from its chunks.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Setup
This notebook uses **one** provider (the first one with a key). If you have no key, it runs in *sample mode* with clearly labelled fake answers, so you can still practise the parsing.

In [ ]:
# ▶ Run this cell first
import os, json, re, time, hashlib, requests
import pandas as pd

def get_key(name):
    """Find a key in Colab Secrets (the 🔑 icon on the left), else in an environment variable.
    A key is never typed into a cell."""
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception:
        return os.environ.get(name) or None

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

# Every provider below speaks the same "chat completions" format, so ONE function can call all of them.
# Model names change often: if you get a 404, run the model list in notebook 05 and pick a current one.
PROVIDERS = {
    "gemini":     {"base": "https://generativelanguage.googleapis.com/v1beta/openai",
                   "key_name": "GEMINI_API_KEY", "model": "gemini-3.8-flash"},
    "groq":       {"base": "https://api.groq.com/openai/v1",
                   "key_name": "GROQ_API_KEY", "model": "llama-3.3-70b-versatile"},
    "openrouter": {"base": "https://openrouter.ai/api/v1",
                   "key_name": "OPENROUTER_API_KEY", "model": "openrouter/free"},
}
ORDER = ["gemini", "groq", "openrouter"]      # the order your app will try them in

def keys_found():
    return [n for n in ORDER if get_key(PROVIDERS[n]["key_name"])]

def redact(text):
    """Hide every key we know about, so messages are safe to show on a projector."""
    text = str(text)
    for p in PROVIDERS.values():
        k = get_key(p["key_name"])
        if k:
            text = text.replace(k, "YOUR_KEY")
    return text

def sample_banner():
    print("⚠️  SAMPLE MODE: this is illustrative text, not a real model answer. Add a free key to see real answers.")

print("Providers with a key:", keys_found() or "none yet. That is OK: the notebook still runs in sample mode.")

In [ ]:
SIMULATE = {}      # planned-failure switch, for example {"gemini": 429, "groq": 401}. Empty = real calls.

def raw_call(name, messages, want_json=False, max_tokens=600, temperature=0.3, timeout=30):
    """ONE attempt at ONE provider. Never raises. Returns a small dict you can inspect."""
    t0 = time.time()
    def fail(status, error, retry_after=None):
        return {"provider": name, "ok": False, "status": status, "error": redact(error)[:200],
                "retry_after": retry_after, "seconds": time.time() - t0}
    if name in SIMULATE:                                   # drill: pretend this provider failed
        return fail(SIMULATE[name], f"simulated {SIMULATE[name]}", 1 if SIMULATE[name] == 429 else None)
    cfg = PROVIDERS[name]
    key = get_key(cfg["key_name"])
    if not key:
        return fail("nokey", "no key set for " + name)
    body = {"model": cfg["model"], "messages": messages, "max_tokens": max_tokens, "temperature": temperature}
    if want_json:
        body["response_format"] = {"type": "json_object"}   # only a hint: always validate what comes back
    headers = {"Authorization": "Bearer " + key, "Content-Type": "application/json"}
    url = cfg["base"] + "/chat/completions"
    try:
        r = requests.post(url, headers=headers, json=body, timeout=timeout)
        if r.status_code == 400 and want_json:             # some models reject response_format: retry without it
            body.pop("response_format")
            r = requests.post(url, headers=headers, json=body, timeout=timeout)
    except requests.Timeout:
        return fail("timeout", "timed out")
    except requests.RequestException as e:
        return fail("network", type(e).__name__)
    if r.status_code != 200:
        try:
            wait = float(r.headers.get("Retry-After"))
        except (TypeError, ValueError):
            wait = None
        return fail(r.status_code, r.text, wait)
    try:
        data = r.json()
        choice = data["choices"][0]
        text = (choice["message"].get("content") or "").strip()
    except (ValueError, KeyError, IndexError, TypeError, AttributeError):
        return fail("badshape", "unexpected response shape")
    if not text:
        return fail("empty", "empty answer")
    return {"provider": name, "ok": True, "status": 200, "text": text,
            "finish_reason": choice.get("finish_reason"), "usage": data.get("usage", {}),
            "seconds": time.time() - t0}

In [ ]:
# 🛟 Safety net for sample mode (no key): ILLUSTRATIVE replies, NOT written by a model.
SAMPLE_TEXT = "SAMPLE (illustrative): It is a rocky leftover from the early solar system that passes close to Earth."
SAMPLE_CUT = "SAMPLE (illustrative): It is a rocky left"
SAMPLE_JSON = """```json
{"headline": "SAMPLE: a quiet flyby", "plain_explanation": "SAMPLE (illustrative): a small rock passes far from Earth.", "risk_level": "low"}
```"""
SAMPLE_SSE = [
    'data: {"choices":[{"delta":{"role":"assistant","content":""}}]}',
    'data: {"choices":[{"delta":{"content":"SAMPLE "}}]}',
    'data: {"choices":[{"delta":{"content":"(illustrative): "}}]}',
    'data: {"choices":[{"delta":{"content":"a streamed answer "}}]}',
    'data: {"choices":[{"delta":{"content":"arrives in pieces."},"finish_reason":"stop"}]}',
    'data: [DONE]',
]

PROVIDER = (keys_found() or ["groq"])[0]
LIVE = bool(keys_found())

def call_llm(messages, want_json=False, max_tokens=600):
    """GIVEN: one call to your provider. Returns the same dict as raw_call. Sample mode when there is no key."""
    if not LIVE:
        sample_banner()
        text = SAMPLE_JSON if want_json else (SAMPLE_CUT if max_tokens < 40 else SAMPLE_TEXT)
        return {"provider": "sample", "ok": True, "status": 200, "text": text,
                "finish_reason": "length" if (max_tokens < 40 and not want_json) else "stop",
                "usage": {"total_tokens": 0}, "seconds": 0.0}
    return raw_call(PROVIDER, messages, want_json=want_json, max_tokens=max_tokens)

def stream_lines(messages):
    """GIVEN: yields the raw lines of a streamed answer (server-sent events)."""
    if not LIVE:
        sample_banner()
        yield from SAMPLE_SSE
        return
    cfg = PROVIDERS[PROVIDER]
    body = {"model": cfg["model"], "messages": messages, "max_tokens": 400, "stream": True}
    with requests.post(cfg["base"] + "/chat/completions", json=body, stream=True, timeout=60,
                       headers={"Authorization": "Bearer " + get_key(cfg["key_name"])}) as r:
        r.raise_for_status()
        for raw in r.iter_lines():
            if raw:
                yield raw.decode("utf-8") if isinstance(raw, bytes) else raw

## 1 · Shape the answer with a system prompt
The `system` message is where you put the rules: audience, length, tone. The `user` message is the question.

Ask for an explanation of why a "near miss" is not a hit, **for a 12-year-old, in two sentences, no jargon**.

### Your turn

In [ ]:
messages = [
    {"role": "system", "content": ...},     # TODO: rules: audience, length, tone
    {"role": "user", "content": "Why is an asteroid that passes at 5 million km not a danger to Earth?"},
]
res = call_llm(messages)
print(res["text"])
print("provider:", res["provider"], "| finish_reason:", res.get("finish_reason"))

In [ ]:
check(lambda: res["ok"] and len(messages[0]["content"]) > 20, "A system prompt shaped the answer.", "Write real rules in the system message.")

## 2 · Detect a cut-off answer
`max_tokens` is a hard limit. If the model hits it, the answer stops mid-sentence and `finish_reason` is `"length"`. Your app must **notice**, or it will show half a sentence. (Some "thinking" models also spend tokens before they answer, so a tiny limit can even give an empty answer.)

Write `is_cut_off(res)`, then ask for a long answer with a tiny limit.

### Your turn

In [ ]:
def is_cut_off(res):
    # TODO: True when the call failed with an empty answer, or finish_reason is "length"
    ...

short = call_llm(messages, max_tokens=20)
print(short.get("text"), "| finish_reason:", short.get("finish_reason"))
print("Cut off?", is_cut_off(short))

In [ ]:
check(lambda: is_cut_off({"ok": True, "status": 200, "finish_reason": "length"}) and not is_cut_off({"ok": True, "status": 200, "finish_reason": "stop"}),
      "Cut-off detection works.", "Return True for finish_reason 'length' and False for 'stop'.")

## 3 · Ask for JSON, then do not trust it
Your code needs fields, not paragraphs. Ask for JSON **in the prompt** (describe the exact keys). Providers also offer a "JSON mode" (`response_format`), but it is only a hint: some models ignore it, and models still wrap JSON in ` ```json ` fences or add chatter.

The rule: **parse defensively, then validate.**

Messy replies you will meet:

In [ ]:
MESSY = [
    '```json\n{"headline": "Quiet flyby", "risk_level": "low"}\n```',
    'Sure! Here is the JSON you asked for:\n{"headline": "Quiet flyby", "risk_level": "low"}\nHope that helps!',
    '{"headline": "Quiet flyby", "risk_level": "low"}',
    "I am sorry, I cannot help with that.",
]
for m in MESSY:
    print(repr(m[:70]))

Write `parse_json(text)`: return a dict, or raise `ValueError` when there is no JSON. Hint: remove ` ``` ` fences, then take the text from the first `{` to the last `}` and give it to `json.loads`.

### Your turn

In [ ]:
def parse_json(text):
    ...      # TODO

for m in MESSY:
    try:
        print("OK  ", parse_json(m))
    except ValueError as e:
        print("FAIL", e)

In [ ]:
def _ok():
    good = [parse_json(m) for m in MESSY[:3]]
    try:
        parse_json(MESSY[3])
        return False
    except ValueError:
        return all(g["risk_level"] == "low" for g in good)
check(_ok, "Three messy replies parsed; the non-JSON reply raised ValueError.",
      "The first three should parse to a dict; the 4th must raise ValueError (json.JSONDecodeError is a ValueError).")

## 4 · Validate, and retry once
Valid JSON can still be wrong: a missing key, or a value outside what your app expects (`risk_level` must be `low`, `medium` or `high`). Write `validate(obj)` and `get_json(messages)`: parse, validate, and on failure **retry once** with a short repair message that shows the model its mistake.

### Your turn

In [ ]:
KEYS = ("headline", "plain_explanation", "risk_level")
LEVELS = ("low", "medium", "high")

def validate(obj):
    # TODO: True only if obj is a dict with all KEYS and risk_level in LEVELS
    ...

def get_json(messages, retries=1):
    msgs = list(messages)
    for attempt in range(retries + 1):
        res = call_llm(msgs, want_json=True)
        if not res["ok"]:
            return None
        try:
            obj = parse_json(res["text"])
            if validate(obj):
                return obj
        except ValueError:
            pass
        # TODO: add the bad reply as an "assistant" message and a "user" message that says the reply was not valid
        #       JSON with exactly these keys, and to reply with JSON only
    return None

In [ ]:
check(lambda: validate({"headline": "a", "plain_explanation": "b", "risk_level": "low"})
      and not validate({"headline": "a"}) and not validate({"headline": "a", "plain_explanation": "b", "risk_level": "huge"}),
      "validate() accepts good objects and rejects bad ones.", "Require all three keys and a risk_level of low, medium or high.")

Now ask for the JSON. Describe the **exact keys and allowed values** in the prompt.

In [ ]:
json_messages = [
    {"role": "system", "content": ...},     # TODO: say: reply with ONE JSON object, keys headline, plain_explanation, risk_level (low|medium|high), no other text
    {"role": "user", "content": "An asteroid about 40 m wide passes at 13 times the distance to the Moon."},
]
card = get_json(json_messages)
print(card)

In [ ]:
check(lambda: card is not None and validate(card), "You got a validated JSON card.", "get_json returned None: check your prompt names the exact keys, and your validate().")

## 5 · Streaming: show words as they arrive
With `"stream": true` the server sends the answer in many small lines (*server-sent events*). Each line looks like `data: {...}`, the new text is in `choices[0].delta.content`, and the last line is `data: [DONE]`. Users see the first words quickly, which feels faster.

Some chunks have no `choices` (for example a final usage chunk), so guard for that.

### Your turn

In [ ]:
def read_stream(lines):
    """Join the pieces of a streamed answer; print each piece as it arrives."""
    text = ""
    for line in lines:
        # TODO: skip lines that do not start with "data:"
        # TODO: stop at "[DONE]"
        # TODO: json.loads the rest, skip chunks without "choices", add choices[0]["delta"].get("content") to text,
        #       and print the piece with print(piece, end="", flush=True)
        ...
    return text

streamed = read_stream(stream_lines(messages))
print()

In [ ]:
check(lambda: read_stream(["data: {\"choices\":[{\"delta\":{\"content\":\"Hi \"}}]}", "data: {\"choices\":[{\"delta\":{\"content\":\"there\"}}]}", "data: [DONE]"]) == "Hi there"
      and len(streamed) > 5, "Stream assembled correctly.", "Join the delta.content pieces, skip non-'data:' lines, stop at [DONE].")

## 6 · Tokens: know what you spend
Free tiers limit tokens per minute and per day. Every response carries `usage`. Print a one-line receipt after each call, and keep prompts short (send the 3 fields you need, not the whole JSON).

### Your turn

In [ ]:
def usage_line(res):
    # TODO: return e.g. "groq: 48 tokens in 0.9 s" using res["provider"], res["usage"]["total_tokens"] and res["seconds"]
    ...

print(usage_line(res))

In [ ]:
check(lambda: "tokens" in usage_line(res), "Receipt printed.", "Return a short string with the provider, the tokens and the seconds.")

## ✅ Done means
- [ ] a system prompt controls audience and length
- [ ] you detect `finish_reason == "length"`
- [ ] a messy reply is parsed, **validated** and retried once
- [ ] a streamed answer is rebuilt from its chunks
- [ ] you can say why JSON mode is "a hint, not a guarantee"

**Next:** notebook `07`: what your app does when a provider says 429, 401 or 503.